# Подготовка данных с финальным target (tp_sl_1_05)

**Шаг 2 плана.** Загрузка финального набора из `02_targets` с уже сохранённым target `tp_sl_1_05` (колонка `target`), валидация и сохранение для следующих ноутбуков `03_features`.

## 1. Импорты

In [1]:
import sys
import os
import numpy as np
import pandas as pd

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.data.dataset_rework_loader import load_dataset_rework
from src.data.data_prep_dataset_rework import prepare_for_training
from src.features.feature_pipeline import add_features, get_feature_columns
from src.features.warmup_loader import add_warmup_from_bybit, remove_warmup

print('Импорты OK')

Импорты OK


## 2. Загрузка данных

In [2]:
prepared_path = os.path.join(_root, 'outputs', 'prepared_with_target_tp_sl_1_05.parquet')
if not os.path.exists(prepared_path):
    raise FileNotFoundError(
        f'Запустите 02_targets/03_Base_Model_And_Target_Comparison.ipynb. Файл не найден: {prepared_path}'
    )

df = pd.read_parquet(prepared_path)
print(f'Загружено: {len(df):,} строк, {df["session_key"].nunique()} сессий')

Загружено: 344,415 строк, 1048 сессий


## 3. Проверка target и распределения

In [3]:
assert 'target' in df.columns, 'Ожидается колонка target в prepared_with_target_tp_sl_1_05.parquet'
assert 'tp_sl_1_05' in df.columns, 'Ожидается колонка tp_sl_1_05 для трассировки происхождения target'

vc = df['target'].dropna().value_counts()
print('Распределение target (tp_sl_1_05, ambiguous intrabar -> 0):')
print(f'  BUY (1):   {vc.get(1.0, 0):,} ({vc.get(1.0, 0)/vc.sum()*100:.1f}%)')
print(f'  SELL (-1): {vc.get(-1.0, 0):,} ({vc.get(-1.0, 0)/vc.sum()*100:.1f}%)')
print(f'  HOLD (0):  {vc.get(0.0, 0):,} ({vc.get(0.0, 0)/vc.sum()*100:.1f}%)')
print(f'Доля NaN target: {df["target"].isna().mean()*100:.2f}%')

Распределение target (tp_sl_1_05, ambiguous intrabar -> 0):
  BUY (1):   97,874 (28.4%)
  SELL (-1): 95,494 (27.7%)
  HOLD (0):  151,047 (43.9%)
Доля NaN target: 0.00%


## 4. Проверка rd_regime и rd_regime_transition

In [4]:
# rd_regime и rd_regime_transition уже в данных как фичи
assert 'rd_regime' in df.columns, 'rd_regime отсутствует'
assert 'rd_regime_transition' in df.columns, 'rd_regime_transition отсутствует'
print('rd_regime:', df['rd_regime'].value_counts().to_dict())
print('rd_regime_transition (доля):', f"{df['rd_regime_transition'].mean():.4f}")

rd_regime: {-1: 172905, 1: 171510}
rd_regime_transition (доля): 0.3807


## 5. Сохранение

In [5]:
out_dir = os.path.join(_root, 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'data_labeled_tp_sl_1_05.parquet')
df_save = df.copy()
df_save['symbol'] = df_save['symbol'].astype(str)
df_save.to_parquet(out_path, index=False, compression='snappy')
print(f'Сохранено: {out_path}')
print(f'Строк: {len(df):,}, сессий: {df["session_key"].nunique()}')

Сохранено: c:\project\trading_bot_2Engine\outputs\data_labeled_tp_sl_1_05.parquet
Строк: 344,415, сессий: 1048


## Итог шага 2

- Загружен финальный датасет с target из `02_targets`
- Проверено распределение `target` (`tp_sl_1_05`, `ambiguous intrabar -> 0`)
- Фичи `rd_regime` и `rd_regime_transition` присутствуют
- Сохранён файл `outputs/data_labeled_tp_sl_1_05.parquet` для ноутбуков 05–09